In [10]:
import pandas as pd, numpy as np
from scipy.spatial.distance import squareform, pdist, cdist
from sklearn.neighbors import kneighbors_graph, NearestNeighbors
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path

In [22]:
def diffusion_map(data, eps, alpha=1, k=4):
    Dsq = squareform(pdist(data)**2)
    Wm = np.exp(-Dsq/eps); q = Wm.sum(1)
    Wa = Wm/np.outer(q**alpha, q**alpha)
    da = Wa.sum(1); Dis = 1.0/np.sqrt(da)
    S = Dis[:, None]*Wa*Dis[None, :]                
    w, v = np.linalg.eigh(S)
    idx = np.argsort(w)[::-1]; w, v = w[idx], v[:, idx]
    phi = Dis[:, None]*v                             
    Psi = phi[:, 1:k+1]*w[1:k+1]                     
    return {"evals": w, "Psi": Psi, "phi": phi, "W": Wm, "degrees": q}

def reconstruct_path(pred, start, end):
    path = [end]; current = end
    while current != start:
        current = pred[start, current]
        if current == -9999: return None
        path.append(current)
    return path[::-1]

def graph_degree_density(Psi, h):
    Dsq = squareform(pdist(Psi)**2)
    rho = np.exp(-Dsq/h).sum(1)
    scale = rho.mean(); rho /= scale
    rho = np.maximum(rho, 1e-6)
    return rho, -np.log(rho), scale

def density_and_bandwidth(Psi, multiplier=0.1):
    median_Dsq = np.median(squareform(pdist(Psi)**2)[np.triu_indices(N,1)])
    h = multiplier * median_Dsq
    return h, graph_degree_density(Psi, h)

def density_aware_cost(A_dist_sym, V, beta):
    rows, cols = A_dist_sym.nonzero()
    base = np.asarray(A_dist_sym[rows, cols]).ravel()
    costs = base*np.exp(beta*(V[rows]+V[cols])/2)
    return csr_matrix((costs, (rows, cols)), shape=A_dist_sym.shape)

def graph_path(A_dist_sym, beta, V, start, end):
    A = density_aware_cost(A_dist_sym, V, beta)
    _, pred = shortest_path(A, directed=False, return_predecessors=True)
    return reconstruct_path(pred, start, end)

def linear_path(Psi, start_idx, end_idx, n_grid=10):
    return np.linspace(Psi[start_idx], Psi[end_idx], num=n_grid)

def local_neighbourhood_lifting(Z, Psi, start_idx, end_idx, m=27, n_grid=10, tau=0.2):
    start_point, end_point = Psi[start_idx], Psi[end_idx]
    gamma = np.linspace(start_point, end_point, num=n_grid)
    nn = NearestNeighbors(n_neighbors=m); nn.fit(Psi)
    distances, indices = nn.kneighbors(gamma)
    a = np.exp(-distances**2/tau) / np.sum(np.exp(-distances**2 / tau), axis=1, keepdims=True)
    points = Z[indices, :]; z_hat = np.sum(a[:, :, None] * points, axis=1)
    z_hat[0], z_hat[-1] = Z[start_idx], Z[end_idx]
    return z_hat, gamma

def latent_density_at_points(gamma, Psi, h, scale):
    Dsq_query = cdist(gamma, Psi, metric="sqeuclidean")
    rho_query = np.exp(-Dsq_query / h).sum(axis=1)
    rho_query /= scale
    rho_query_floor = np.maximum(rho_query, 1e-6)
    V_query = -np.log(rho_query_floor)
    return rho_query_floor, V_query

def lift_validation(z_hat, Z):
    nn = NearestNeighbors(n_neighbors=1); nn.fit(Z)
    dNN, _ = nn.kneighbors(z_hat)
    return dNN

Needed
- Embedding $\Psi$
- Key dates
- KNN graph and cost graph for diffusion embedding
- Graph paths (standard and density-aware)
- Density in diffusion coordinates for linear path
- Local-neighbourhood lifting
- Nearest neighbour of lifted coordinates


After all functions are imported
- For each pair of endpoints, construct 3 different paths 
    - Need Psi, endpoints, store paths in dictionary: `path_dict = {"GFC": {"Linear": [...], "graph": [...], "density-aware": [...]}, ...}`

In [13]:
K_GRAPH = 15
BETA = 0.75

In [ ]:
df = pd.read_parquet("./datasets/joint_df_quantile.parquet")
dates = df.index; variables = df.columns[:-1]
Z = df.to_numpy()[:, :-1]; N = Z.shape[0]
TAU = np.median(squareform(pdist(Z)**2)[np.triu_indices(N,1)])

diff = diffusion_map(Z, eps=3, k=3); Psi = diff["Psi"]

endpoint_pairs = {"2006 benign -> 2008 GFC": ("2006-03-01", "2008-10-01"), "2019 benign -> 2020 COVID": ("2019-07-01", "2020-04-01"), 
                  "2019 benign -> 2021 Fiscal Tightening": ("2019-04-01", "2022-04-01"), "1977 benign -> 1982 Recession Trough": ("1977-01-01", "1982-07-01")}
endpoints = {name: (dates.get_loc(pair[0]), dates.get_loc(pair[1])) for name, pair in endpoint_pairs.items()}

A_dist = kneighbors_graph(Psi, n_neighbors=K_GRAPH, mode="distance", include_self=False)
A_dist_sym = A_dist.maximum(A_dist.T)

h_dens, (rho, V, scale) = density_and_bandwidth(Psi)

### Finding 3 paths for each pair of endpoints
I also find the density of each path here

In [39]:
path_dict = {}

for event, (start_idx, end_idx) in endpoints.items():
    path_lin = linear_path(Psi, start_idx, end_idx)
    path_lin_density = latent_density_at_points(path_lin, Psi, h_dens, scale)[0]

    path_graph = graph_path(A_dist_sym, beta=0.0, V=V, start=start_idx, end=end_idx)
    path_graph_density = rho[path_graph][1:-1]

    path_dense = graph_path(A_dist_sym, beta=0.75, V=V, start=start_idx, end=end_idx)
    path_dense_density = rho[path_dense][1:-1]

    path_dict[event] = {"linear": path_lin, "linear_density": path_lin_density,
                        "graph_path": path_graph, "graph_path_density": path_graph_density,
                        "density_aware_path": path_dense, "dense_aware_path_density": path_dense_density}

Work on one metric at a time

In [42]:
# minimum densty along the path
rows = []

for event, data in path_dict.items():
    min_d_lin = data["linear_density"].min()
    min_d_graph = data["graph_path_density"].min()
    min_d_aware = data["dense_aware_path_density"].min()

    rows.append((event, min_d_lin, min_d_graph, min_d_aware))

df = pd.DataFrame(rows, columns=["event", "min_d_lin", "min_d_graph", "min_d_aware"])
df

,event,min_d_lin,min_d_graph,min_d_aware
0,2006 benign -> 2008 GFC,0.000973,0.170639,0.311952
1,2019 benign -> 2020 COVID,0.049545,0.186054,0.240622
2,2019 benign -> 2021 Fiscal Tightening,0.382413,0.851436,0.918198
3,1977 benign -> 1982 Recession Trough,0.123264,0.344411,0.344411
